# Đánh giá M0–M3

Chọn `TASK='validation'` để sinh prediction hoặc `TASK='semantic'` để tính BERTScore từ prediction đã đủ.
Notebook không train, không mở test và không thay notebook lịch sử.
Mặc định chỉ kiểm đầu vào; bật `RUN_EXPORT` sau khi kiểm đạt.


In [ ]:
import hashlib
import json
import os
import re
import runpy
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = os.environ.get('VQA_PROJECT_ROOT')
VALIDATION_RECORD = ''  # Chọn record mới đã kiểm và source hỗ trợ tác vụ
VALIDATION_SHA256 = ''
EXPECTED_SEED = 42
TASK = 'validation'
ANALYSIS_RELEASE = 'source_20261020_074604_501083'
ANALYSIS_SHA256 = 'ac1cf60cef5dfcbc9dfc7fd66ae5d5dd3f2def3efc440027aabeadf93982c08b'
SEMANTIC_OUTPUT = ''  # Chọn thư mục riêng cho lượt semantic
RUN_EXPORT = False
INSPECTED = False


## 1. Kho dự án và môi trường

Validation dùng source suy luận đã ghim trong record. Semantic dùng source phân tích riêng và venv riêng, không nạp Qwen hoặc tải lại ảnh.


In [ ]:
from google.colab import drive

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive', timeout_ms=600000)
if PROJECT_DIR:
    ROOT = Path(PROJECT_DIR)
else:
    candidates = [Path('/content/drive/MyDrive/Vietnamese_VQA_NLE')]
    candidates += list(Path('/content/drive/.shortcut-targets-by-id').glob('*/Vietnamese_VQA_NLE'))
    roots = {p.resolve() for p in candidates
             if (p / 'reports/m1_selection/frozen_checkpoint3683_v2/selection.json').is_file()}
    if len(roots) != 1:
        raise ValueError('Set PROJECT_DIR to the mounted project folder; no unique match found')
    ROOT = roots.pop()
if not ROOT.is_dir():
    raise FileNotFoundError(ROOT)
print('Project:', ROOT)


In [ ]:
INSPECTED = False
if TASK not in {'validation', 'semantic'}:
    raise ValueError('Choose validation or semantic')
if not VALIDATION_RECORD or not re.fullmatch(r'[0-9a-f]{64}', VALIDATION_SHA256 or ''):
    raise ValueError('Select a validation launch record and its SHA-256')
record_path = ROOT / VALIDATION_RECORD
if hashlib.sha256(record_path.read_bytes()).hexdigest() != VALIDATION_SHA256:
    raise ValueError('Validation launch changed')
record = json.loads(record_path.read_text('utf-8'))
if (record.get('schema') != 'preference_validation_launch_v1'
        or record.get('scope') != 'validation_only'
        or record.get('training_seed') != EXPECTED_SEED
        or record.get('questions_per_arm') != 1407):
    raise ValueError('Unexpected validation launch')
training_path = ROOT / record['training_launch']
if hashlib.sha256(training_path.read_bytes()).hexdigest() != record['training_launch_sha256']:
    raise ValueError('Training launch changed')
launch = json.loads(training_path.read_text('utf-8'))
if (launch.get('data_locked') is not True or launch.get('expected_pairs') != 1000
        or launch.get('training_seed', 42) != EXPECTED_SEED
        or launch['run_group'] != record['run_group']):
    raise ValueError('Validation points to a different training run')

bootstrap = ROOT / 'source/bootstrap_cloud_source.py'
if hashlib.sha256(bootstrap.read_bytes()).hexdigest() != (
    '391172ed1fbf39d0eeeef2c9aa3d056d7770085d0a10574d571470d7f2bbef40'
):
    raise ValueError('Bootstrap checksum mismatch')
release = record['source_release'] if TASK == 'validation' else ANALYSIS_RELEASE
source_sha256 = record['source_sha256'] if TASK == 'validation' else ANALYSIS_SHA256
if not isinstance(release, str) or not re.fullmatch(r'[0-9a-f]{64}', source_sha256 or ''):
    raise ValueError('Select a verified analysis release and its SHA-256')
if not re.fullmatch(r'source_[A-Za-z0-9_]+', release):
    raise ValueError('Invalid source release')
REPO = Path(subprocess.check_output([
    sys.executable, str(bootstrap), '--archive',
    str(ROOT / 'source/releases' / f'{release}.zip'),
    '--sha256', source_sha256, '--work-root', '/content/vqa_sources',
], text=True).strip())

run_command = runpy.run_path(str(REPO / 'src/vqa_explanation/utils/process.py'))['run_command']

def run(command):
    run_command([str(part) for part in command], cwd=REPO)

ENV_DIR = Path('/content/vqa_envs') / (release + ('_semantic' if TASK == 'semantic' else ''))
PYTHON = ENV_DIR / 'bin/python'
if TASK == 'semantic':
    run([sys.executable, REPO / 'scripts/setup_cloud_environment.py',
         '--profile', 'semantic', '--environment-dir', ENV_DIR])
else:
    run([sys.executable, REPO / 'scripts/setup_preference_environment.py',
         '--environment-dir', ENV_DIR])
print('Source:', release)
print('Python:', PYTHON)
OUTPUT = ROOT / record['output_dir']
print('Validation seed:', EXPECTED_SEED)
print('Output:', OUTPUT)


In [ ]:
INSPECTED = False
if TASK == 'semantic':
    if not SEMANTIC_OUTPUT:
        raise ValueError('Choose a semantic output directory for the selected validation')
    ANNOTATIONS = ROOT / 'reports/m1_selection/frozen_checkpoint3683_v2/validation.jsonl'
    if not ANNOTATIONS.is_file():
        raise FileNotFoundError('Frozen validation annotations are missing')
    SEMANTIC_DIR = ROOT / SEMANTIC_OUTPUT
else:
    ENVIRONMENT = REPO / launch['environment']
    paths_code = (
        'import json,sys; from pathlib import Path; '
        'from vqa_explanation.utils.environment import load_environment; '
        'print(json.dumps({k:str(v) for k,v in load_environment(Path(sys.argv[1])).items()}))'
    )
    child_env = {k: v for k, v in os.environ.items() if k not in {'PYTHONPATH', 'PYTHONHOME'}}
    child_env['PYTHONNOUSERSITE'] = '1'
    paths = json.loads(subprocess.check_output(
        [str(PYTHON), '-c', paths_code, str(ENVIRONMENT)], cwd=REPO, text=True, env=child_env,
    ))
    bundle = ROOT / launch['dataset_bundle']
    if hashlib.sha256((bundle / 'bundle.json').read_bytes()).hexdigest() != (
        launch['dataset_bundle_sha256']
    ):
        raise ValueError('Dataset bundle changed')
    if not Path(paths['annotations']).is_file() or not Path(paths['image_checksums']).is_file():
        run([PYTHON, REPO / 'scripts/verify_coco_bundle.py', bundle,
             '--extract-to', paths['image_root']])


## 2. Kiểm đầu vào

Validation kiểm lượt train và manifest trước khi nối tiếp. Nếu record ghim baseline, CLI kiểm đủ M0/M1 và identity trước khi sao chép; chỉ sinh prediction M2/M3 của lượt mới. Semantic kiểm đủ 1.407 câu mỗi nhánh, ID, tham chiếu, manifest và cùng giao thức; không sinh prediction mới.


In [ ]:
INSPECTED = False
if TASK == 'semantic':
    VALIDATION_COMMAND = [
        str(PYTHON), '-u', str(REPO / 'scripts/run_study_semantic.py'),
        '--results', str(OUTPUT), '--annotations', str(ANNOTATIONS),
        '--output-dir', str(SEMANTIC_DIR), '--device', 'cuda', '--batch-size', '16',
        '--cache-dir', '/content/vqa_semantic_cache',
    ]
else:
    VALIDATION_COMMAND = [
        str(PYTHON), '-u', str(REPO / 'scripts/run_study_validation.py'),
        '--environment', str(ENVIRONMENT), '--study-config', str(REPO / launch['study_config']),
        '--run-group', str(ROOT / record['run_group']),
        '--m1-adapter', str(ROOT / launch['m1_adapter']), '--output-dir', str(OUTPUT),
        '--m2-stage', record['m2_stage'], '--m3-stage', record['m3_stage'],
    ]
    if 'baseline_source' in record or 'baseline_sha256' in record:
        VALIDATION_COMMAND += [
            '--baseline-source', str(ROOT / record['baseline_source']),
            '--baseline-sha256', json.dumps(record['baseline_sha256']),
        ]

run(VALIDATION_COMMAND + ['--inspect'])
INSPECTED = True


## 3. Chạy tác vụ đã chọn

Bật `RUN_EXPORT` sau khi inspect đạt. BERTScore lưu riêng từng nhánh và chỉ tính nhánh còn thiếu; file đã có phải khớp prediction và giao thức. Không ghi đè hoặc trộn kết quả cũ.


In [ ]:
if not INSPECTED:
    raise RuntimeError('Run validation inspection first')
if RUN_EXPORT:
    run(VALIDATION_COMMAND)
else:
    print('Prediction export not requested')


## 4. Kết quả

Report một seed dùng `scripts/report_study_validation.py`; tổng hợp ba seed dùng `scripts/report_validation_seeds.py`. Chỉ tổng hợp khi đủ cả ba bảng và BERTScore cùng giao thức. Test/chấm mù theo ảnh là bước riêng.


In [ ]:
if TASK == 'semantic':
    receipt = SEMANTIC_DIR / 'completed.json'
    if receipt.is_file():
        print(json.dumps(json.loads(receipt.read_text('utf-8')), indent=2))
    else:
        print('BERTScore not completed:', SEMANTIC_DIR)
else:
    print('Result folder:', OUTPUT)
    for arm in ('m0', 'm1', 'm2', 'm3'):
        prediction = OUTPUT / f'{arm}.jsonl'
        if prediction.is_file():
            with prediction.open(encoding='utf-8') as handle:
                count = sum(bool(line.strip()) for line in handle)
            print(arm.upper(), 'saved rows:', count)
        else:
            print(arm.upper(), 'prediction file missing')
    print('Row counts alone do not certify a complete or comparable export.')
